# Tokenización y Representaciones Vectoriales en LLMs

## ¿Qué es la tokenización?

Antes de que un modelo de lenguaje como GPT o BERT pueda procesar texto, necesita convertir las palabras (o partes de palabras) en números. Este proceso se llama **tokenización**.

Un **token** es la unidad básica de texto que el modelo "ve". Puede ser:
- Una palabra completa: `"perro"` → token
- Un subpalabra: `"tokenización"` → `["token", "ización"]`
- Un carácter individual

## ¿Qué es un embedding (representación vectorial)?

Una vez tokenizado el texto, cada token se mapea a un **vector de números reales** en un espacio de alta dimensión. Este vector es el *embedding* del token.

**¿Por qué vectores?** Porque nos permiten representar el *significado semántico* de las palabras de forma matemática. Palabras con significados similares quedan **cerca** en este espacio vectorial.

```
"rey"   → [0.23, -0.81, 0.45, ..., 0.12]  (300 números)
"reina" → [0.21, -0.79, 0.48, ..., 0.15]  (300 números, ¡muy similar!)
"mesa"  → [-0.54, 0.33, -0.12, ..., 0.87] (300 números, muy diferente)
```

En este notebook usaremos **Word2Vec** (Google News), un modelo clásico entrenado sobre millones de artículos de noticias, para explorar estas representaciones.

### Nota de la solución añadida

Se conservan todos los Markdown de la guía, su secuencia y los identificadores originales. [Fuente exacta](https://github.com/EAFIT-IA/si3003-artificial-intelligence/blob/dfbcd874eb8bc27a3f3c258b0e7fdc41a6d9dbd3/notebooks/Lecture9/tokenization_Introduction.ipynb).

El reto se completa en su celda original. Los ejemplos permanecen y se protegen para validar sin descargar. Las afirmaciones empíricas escritas en la guía (como una posición de `queen`) son texto del autor: **no son resultados de esta entrega**. Se limpiaron las salidas previas.

Para ejecutar todo: prepara dependencias y habilita la carga del modelo abajo. Google News requiere aproximadamente 1,6 GB de descarga y varios GB de RAM; sus vectores float32 ocupan unos 3,6 GB, más objetos y temporales. No requiere GPU. En Colab, reinicia el kernel tras actualizar bibliotecas ya importadas y guarda los resultados antes de terminar la sesión.

## Paso 1 — Instalar dependencias

`gensim` es una librería de Python especializada en modelos de lenguaje y representaciones vectoriales. La usaremos para cargar modelos Word2Vec pre-entrenados sin necesidad de entrenar desde cero.

In [ ]:
INSTALAR_DEPENDENCIAS = False  # True solo para preparar el entorno.

In [ ]:
# Instalación original convertida en una opción explícita.
if INSTALAR_DEPENDENCIAS:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "gensim>=4.3,<5", "numpy", "matplotlib", "scikit-learn"])
    print("Si había bibliotecas importadas, reinicia el kernel y ejecuta de nuevo.")
else:
    print("Instalación omitida: se usarán las dependencias disponibles.")

### Configuración y utilidades añadidas

Sin descarga por defecto. Para resultados reales, indica un archivo confiable en `RUTA_LOCAL` o cambia `PERMITIR_DESCARGA=True`. El modelo de la guía es `word2vec-google-news-300`. La alternativa explícita `glove-wiki-gigaword-50` **no equivale al entregable Word2Vec**.

La suma/resta usa vectores originales del mismo modelo; el coseno normaliza consulta y candidatos. Se excluyen los términos fuente en el reto. No se sustituyen palabras ausentes ni se cambia su capitalización. La exportación distingue modelo y ejecución (fecha UTC + UUID). Los archivos nativos Gensim `.kv` deben ser confiables.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import csv
import json
import platform
import re
import numpy as np
import matplotlib.pyplot as plt

# El modelo oficial de la guía. No hay cambio automático a otro modelo.
MODELO_ID = "word2vec-google-news-300"
PERMITIR_DESCARGA = False
RUTA_LOCAL = ""  # Ejemplo: "/content/GoogleNews-vectors-negative300.bin.gz"
FORMATO_LOCAL = "word2vec_bin"  # "gensim_kv", "word2vec_bin" o "word2vec_text"
DESCRIPCION_LOCAL = "Archivo local: procedencia declarada por quien lo aporta"

ANIMAL = "tiger"
TOP_N = 10
BLOQUE_BUSQUEDA = 25_000  # Evita normalizar/copiar todo el vocabulario a la vez.
DIRECTORIO_SALIDA = Path.cwd() / "resultados_word2vec_guia"

ANALOGIAS = [
    {
        "nombre": "tiger_wild_domestic",
        "positivos": ["tiger", "domestic"],
        "negativos": ["wild"],
        "expresion": "tiger - wild + domestic",
        "esperado": "cat",
        "hipotesis": "Al sustituir la asociación con lo salvaje por lo doméstico, espero un felino doméstico como cat."
    },
    {
        "nombre": "puppy_dog_cat",
        "positivos": ["puppy", "cat"],
        "negativos": ["dog"],
        "expresion": "puppy - dog + cat",
        "esperado": "kitten",
        "hipotesis": "Espero trasladar la relación cría/adulto de perro a gato y encontrar kitten."
    },
]
print("Configuración: sin descarga por defecto; resultados reales solo al cargar un modelo.")

In [ ]:
def normalizar(vector):
    """Vector unitario; rechaza ceros, dimensiones incorrectas y no finitos."""
    vector = np.asarray(vector, dtype=np.float64)
    if vector.ndim != 1 or not np.all(np.isfinite(vector)):
        raise ValueError("Se necesita un vector 1D con valores finitos.")
    norma = np.linalg.norm(vector)
    if norma <= np.finfo(float).eps:
        raise ValueError("No puede normalizarse un vector de norma cero.")
    return vector / norma


def coseno(a, b):
    a, b = normalizar(a), normalizar(b)
    if a.shape != b.shape:
        raise ValueError("Los vectores deben pertenecer al mismo espacio.")
    return float(np.clip(a @ b, -1.0, 1.0))


def faltantes(modelo, palabras):
    """Mantiene las palabras exactas: no cambia mayúsculas silenciosamente."""
    return [palabra for palabra in dict.fromkeys(palabras)
            if palabra not in modelo.key_to_index]


def vector_analogia(modelo, positivos, negativos):
    if not positivos:
        raise ValueError("La analogía necesita al menos un término positivo.")
    ausentes = faltantes(modelo, [*positivos, *negativos])
    if ausentes:
        raise KeyError("Palabras fuera del vocabulario: " + ", ".join(ausentes))
    resultado = np.zeros(modelo.vector_size, dtype=np.float64)
    for palabra in positivos:
        resultado += np.asarray(modelo[palabra], dtype=np.float64)
    for palabra in negativos:
        resultado -= np.asarray(modelo[palabra], dtype=np.float64)
    normalizar(resultado)  # Validación de norma; devolver la suma ORIGINAL.
    return resultado


def vecinos_coseno(modelo, consulta, topn=10, excluir=(), bloque=25_000):
    """Búsqueda exacta por bloques, sin crear otra matriz normalizada completa."""
    if topn < 1 or bloque < 1:
        raise ValueError("topn y bloque deben ser positivos.")
    consulta = normalizar(consulta)
    if consulta.shape != (modelo.vector_size,):
        raise ValueError("Dimensión de consulta incompatible con el modelo.")
    excluidos = {modelo.key_to_index[p] for p in excluir if p in modelo.key_to_index}
    mejores = []
    cantidad = len(modelo.index_to_key)
    for inicio in range(0, cantidad, bloque):
        final = min(inicio + bloque, cantidad)
        vectores = np.asarray(modelo.vectors[inicio:final])
        normas = np.linalg.norm(vectores, axis=1)
        puntuaciones = np.full(final - inicio, -np.inf, dtype=np.float64)
        validos = np.isfinite(normas) & (normas > 0)
        puntuaciones[validos] = (vectores[validos] @ consulta) / normas[validos]
        puntuaciones[~np.isfinite(puntuaciones)] = -np.inf
        for indice in excluidos:
            if inicio <= indice < final:
                puntuaciones[indice - inicio] = -np.inf
        # Orden estable: conserva el menor índice incluso en empates al corte.
        candidatos = np.argsort(-puntuaciones, kind="stable")[:topn]
        mejores.extend(
            (float(np.clip(puntuaciones[i], -1, 1)), inicio + int(i))
            for i in candidatos if np.isfinite(puntuaciones[i])
        )
        # Desempates por índice del vocabulario: reproducibles para este archivo.
        mejores = sorted(mejores, key=lambda item: (-item[0], item[1]))[:topn]
    return [{"palabra": modelo.index_to_key[indice], "coseno": puntuacion}
            for puntuacion, indice in mejores]


def pca_tres_dimensiones(vectores):
    """PCA por SVD de una matriz pequeña: coordenadas N×3 y varianza explicada."""
    matriz = np.asarray(vectores, dtype=np.float64)
    if matriz.ndim != 2 or min(matriz.shape) < 3 or matriz.shape[0] < 4:
        raise ValueError("PCA 3D necesita al menos 4 puntos y 3 atributos.")
    if not np.all(np.isfinite(matriz)):
        raise ValueError("PCA no admite valores no finitos.")
    centrada = matriz - matriz.mean(axis=0)
    u, singular, _ = np.linalg.svd(centrada, full_matrices=False)
    energia = singular ** 2
    if energia.sum() <= np.finfo(float).eps:
        raise ValueError("PCA requiere variación entre los puntos.")
    coordenadas = u[:, :3] * singular[:3]
    explicada = energia[:3] / energia.sum()
    return coordenadas, explicada

### Pruebas offline

Vectores construidos a mano: validan aritmética, empates, exclusiones y PCA. No son resultados de Google News.

In [ ]:
class VectoresSinteticos:
    def __init__(self, datos):
        self.index_to_key = list(datos)
        self.key_to_index = {palabra: i for i, palabra in enumerate(datos)}
        self.vectors = np.asarray(list(datos.values()), dtype=np.float64)
        self.vector_size = self.vectors.shape[1]

    def __getitem__(self, palabra):
        return self.vectors[self.key_to_index[palabra]]


def pruebas_offline():
    prueba = VectoresSinteticos({
        "dog": [1, 0, 0, 0], "puppy": [1, 1, 0, 0],
        "cat": [0, 0, 1, 0], "kitten": [0, 1, 1, 0],
        "other": [-1, 0, 0, 1],
    })
    operacion = vector_analogia(prueba, ["puppy", "cat"], ["dog"])
    assert np.allclose(operacion, prueba["kitten"])
    candidatos = vecinos_coseno(
        prueba, operacion, topn=2, excluir=["puppy", "dog", "cat"], bloque=2
    )
    assert candidatos[0]["palabra"] == "kitten"
    assert np.isclose(candidatos[0]["coseno"], 1)
    assert not {"puppy", "dog", "cat"} & {r["palabra"] for r in candidatos}
    # El top-N no cambia con el tamaño de bloque aunque haya empates exactos.
    empatados = VectoresSinteticos({
        "primero": [1, 0, 0], "segundo": [1, 0, 0],
        "tercero": [1, 0, 0], "cuarto": [1, 0, 0],
        "ultimo": [0, 1, 0],
    })
    for tamano_bloque in (1, 3, 5):
        top = vecinos_coseno(empatados, [1, 0, 0], topn=2, bloque=tamano_bloque)
        assert [fila["palabra"] for fila in top] == ["primero", "segundo"]
        top_filtrado = vecinos_coseno(
            empatados, [1, 0, 0], topn=2, excluir=["primero"], bloque=tamano_bloque
        )
        assert [fila["palabra"] for fila in top_filtrado] == ["segundo", "tercero"]
    assert faltantes(prueba, ["cat", "Cat", "Cat"]) == ["Cat"]
    assert np.isclose(coseno([1, 0], [-1, 0]), -1)
    try:
        vector_analogia(prueba, ["unknown"], ["dog"])
    except KeyError:
        pass
    else:
        raise AssertionError("Debió informar la palabra ausente.")
    try:
        normalizar([0, 0])
    except ValueError:
        pass
    else:
        raise AssertionError("Debió rechazar la norma cero.")
    coordenadas, explicada = pca_tres_dimensiones(prueba.vectors)
    assert coordenadas.shape == (5, 3) and explicada.shape == (3,)
    assert np.isfinite(coordenadas).all()
    assert 0 < explicada.sum() <= 1 + 1e-12
    # PCA de cuatro puntos tiene como máximo tres direcciones no nulas.
    cuatro = prueba.vectors[:4]
    reducidos, _ = pca_tres_dimensiones(cuatro)
    assert np.allclose(
        np.linalg.norm(cuatro[:, None] - cuatro[None, :], axis=2),
        np.linalg.norm(reducidos[:, None] - reducidos[None, :], axis=2)
    )
    print("Pruebas offline aprobadas. Vectores artificiales; NO resultados de Google News.")


pruebas_offline()

## Paso 2 — Cargar el modelo Word2Vec

Cargamos el modelo `word2vec-google-news-300`, entrenado por Google sobre ~100 mil millones de palabras de artículos de noticias.

- Contiene ~3 millones de palabras/frases en su vocabulario
- Cada token está representado por un vector de **300 dimensiones**

> **Nota:** La primera vez esto descarga ~1.6 GB. Después queda en caché local.

In [ ]:
def cargar_modelo():
    ruta = Path(RUTA_LOCAL).expanduser() if RUTA_LOCAL else None
    if ruta is None and not PERMITIR_DESCARGA:
        return None, {
            "estado": "pendiente_modelo",
            "mensaje": "Sin modelo: indica RUTA_LOCAL o permite la descarga explícitamente.",
            "modelo_solicitado": MODELO_ID,
        }
    try:
        import gensim
        from gensim.models import KeyedVectors
    except ImportError as error:
        raise ImportError(
            "Se requiere Gensim para cargar el modelo. Usa la instalación opcional "
            "y reinicia el kernel si es necesario."
        ) from error

    if ruta is not None:
        if not ruta.is_file():
            raise FileNotFoundError(f"No existe el archivo local: {ruta}")
        if FORMATO_LOCAL == "gensim_kv":
            modelo = KeyedVectors.load(str(ruta), mmap=None)
        elif FORMATO_LOCAL in {"word2vec_bin", "word2vec_text"}:
            modelo = KeyedVectors.load_word2vec_format(
                str(ruta), binary=(FORMATO_LOCAL == "word2vec_bin")
            )
        else:
            raise ValueError("FORMATO_LOCAL no reconocido.")
        origen = {
            "modelo": "archivo_local", "procedencia": DESCRIPCION_LOCAL,
            "archivo": ruta.name, "formato": FORMATO_LOCAL,
            "es_modelo_oficial_verificado": False,
        }
    else:
        permitidos = {"word2vec-google-news-300", "glove-wiki-gigaword-50"}
        if MODELO_ID not in permitidos:
            raise ValueError("Selecciona uno de los dos modelos documentados.")
        import gensim.downloader as api
        print(f"Cargando {MODELO_ID}; puede descargar datos si no están en caché.")
        modelo = api.load(MODELO_ID)
        origen = {
            "modelo": MODELO_ID, "procedencia": "gensim.downloader",
            "es_modelo_oficial_verificado": MODELO_ID == "word2vec-google-news-300",
        }
        dimension_esperada = 300 if MODELO_ID == "word2vec-google-news-300" else 50
        if modelo.vector_size != dimension_esperada:
            raise ValueError("Dimensión inesperada para el modelo elegido.")
    origen.update({
        "estado": "cargado", "dimensiones_embedding": int(modelo.vector_size),
        "vocabulario": int(len(modelo.index_to_key)), "gensim": gensim.__version__,
        "numpy": np.__version__, "python": platform.python_version(),
    })
    return modelo, origen


modelo, origen = cargar_modelo()
model = modelo  # Nombre usado en las celdas originales.
print(json.dumps(origen, ensure_ascii=False, indent=2))

## Paso 3 — ¿Cuántas dimensiones tiene un embedding?

Verifiquemos la dimensión del vector de la palabra `"apple"`. Cada token en este modelo está representado como un punto en un espacio de **300 dimensiones**.

Aunque 300 dimensiones es imposible de visualizar directamente, podemos usar técnicas de reducción dimensional como **PCA** (Análisis de Componentes Principales) o **t-SNE** para proyectarlos en 2D o 3D y así visualizar las relaciones semánticas entre palabras.

In [ ]:
if model is None:
    print("Dimensión pendiente: no se ha cargado el modelo.")
elif "apple" not in model.key_to_index:
    print("'apple' no está en el vocabulario de este archivo.")
else:
    print("Dimensiones del vector de apple:", len(model["apple"]))
    print("Dimensiones declaradas por el modelo:", model.vector_size)

## Paso 4 — Aritmética vectorial: la magia de los embeddings

Una de las propiedades más fascinantes de los embeddings es que **las relaciones semánticas se preservan como operaciones vectoriales**.

El experimento clásico es:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

¿Qué significa esto? El vector que va de `"man"` a `"king"` (la *dirección de realeza*) es aproximadamente el mismo que el que va de `"woman"` a `"queen"`. El modelo aprendió este concepto implícitamente al leer millones de textos — **nadie le enseñó explícitamente qué es un rey o una reina**.

### Visualización con PCA 3D

Como no podemos graficar 300 dimensiones, usamos **PCA** para reducir a 3D, buscando las tres direcciones que capturan la mayor varianza. Las líneas rojas y azules muestran la similitud coseno entre el vector resultado y `king`/`queen`.

In [ ]:
v_king = v_man = v_woman = v_queen = v_result = None
if model is None:
    print("Ejemplo de analogía pendiente: modelo no cargado.")
elif faltantes(model, ["king", "man", "woman", "queen"]):
    print("Ejemplo omitido; faltan:", faltantes(model, ["king", "man", "woman", "queen"]))
else:
    from sklearn.decomposition import PCA
    from sklearn.metrics.pairwise import cosine_similarity
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    import numpy as np
    
    
    v_king = model['king']
    v_man = model['man']
    v_woman = model['woman']
    v_queen = model['queen']
    v_result = v_king - v_man + v_woman
    
    
    words_to_plot = ['king', 'man', 'queen', 'woman', 'result']
    vectors_to_plot = [v_king, v_man, v_queen, v_woman, v_result]
    
    pca = PCA(n_components=3)
    reduced = pca.fit_transform(vectors_to_plot)
    coords = {word: reduced[i] for i, word in enumerate(words_to_plot)}
    
    def cos(a, b):
        return cosine_similarity([a], [b])[0, 0]
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    for word, vec in coords.items():
        ax.scatter(vec[0], vec[1], vec[2])
        ax.text(vec[0], vec[1], vec[2], word)
    
    result = coords['result']
    king = coords['king']
    queen = coords['queen']
    
    # Result ↔ King
    ax.plot([result[0], king[0]], [result[1], king[1]], [result[2], king[2]], color='red')
    mid_rk = (result + king) / 2.0
    ax.text(mid_rk[0], mid_rk[1], mid_rk[2],
            f"sim={cos(v_result, v_king):.2f}", color='red')
    
    # King ↔ Queen
    ax.plot([king[0], queen[0]], [king[1], queen[1]], [king[2], queen[2]], color='red')
    mid_kq = (king + queen) / 2.0
    ax.text(mid_kq[0], mid_kq[1], mid_kq[2],
            f"sim={cos(v_king, v_queen):.2f}", color='red')
    
    # Result ↔ Queen
    ax.plot([result[0], queen[0]], [result[1], queen[1]], [result[2], queen[2]], color='blue')
    mid_rq = (result + queen) / 2.0
    ax.text(mid_rq[0], mid_rq[1], mid_rq[2],
            f"sim={cos(v_result, v_queen):.2f}", color='blue')
    
    ax.set_title("3D PCA — King - Man + Woman = Result")
    ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2"); ax.set_zlabel("PCA 3")
    plt.show()
    plt.close(fig)

## Paso 5 — ¿A qué palabras se parece el resultado?

Ahora buscamos los 10 tokens más similares al vector resultado de `king - man + woman` usando **similitud coseno**.

La similitud coseno mide el ángulo entre dos vectores en el espacio de embeddings:
- `sim = 1.0` → misma dirección (palabras idénticas o sinónimos perfectos)
- `sim ≈ 0.7` → fuertemente relacionadas
- `sim ≈ 0.0` → sin relación semántica

Si los embeddings capturaron bien la semántica, deberíamos ver `"queen"` entre los primeros resultados — y de hecho aparece en el segundo lugar.

In [ ]:
if model is None or v_result is None:
    print("Consulta pendiente: modelo o vector del ejemplo no disponible.")
else:
    similar_words = model.most_similar(positive=[v_result], topn=10)
    
    print("Top 10 palabras mas similares a la operación 'king - man + woman':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

### Vecinos más cercanos de `"queen"`

Para comparar, veamos directamente los 10 tokens más similares a `"queen"`. Esto nos dice qué conceptos el modelo asocia con ser reina. Observa que `king` aparece como vecino de `queen` y viceversa — los embeddings capturaron la relación bidireccional entre estos conceptos.

In [ ]:
if model is None or v_queen is None:
    print("Consulta pendiente: modelo o vector del ejemplo no disponible.")
else:
    similar_words = model.most_similar(positive=[v_queen], topn=10)
    
    print("Top 10 palabras mas similares a la operación 'queen':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

### Vecinos más cercanos de `"king"`

Finalmente, los 10 tokens más similares a `"king"`. Compara los vecinos de `king` vs. los de `queen`: ¿son simétricos? ¿Hay sesgos del corpus de entrenamiento que puedas identificar?

### Respuesta añadida a la comparación

El coseno entre dos vectores es simétrico, pero pertenecer al **top-10** del otro no tiene por qué serlo: cada palabra compite con otros vecinos. Polisemia, nombres propios, frecuencia y asociaciones del corpus influyen en las listas. Las celdas originales pasan vectores a `most_similar`, por lo que pueden incluir la palabra consultada; en el reto la excluimos.

Coseno 1 significa la misma dirección, no necesariamente sinonimia; 0 significa ortogonalidad, no ausencia universal de relación lingüística. Los umbrales ilustrativos del Markdown original no son reglas semánticas universales.

In [ ]:
if model is None or v_king is None:
    print("Consulta pendiente: modelo o vector del ejemplo no disponible.")
else:
    similar_words = model.most_similar(positive=[v_king], topn=10)
    
    print("Top 10 palabras mas similares a la operación 'king':")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")

---
## Reto: Explora tu propio espacio semántico

Ahora es tu turno. Usando el modelo Word2Vec que ya cargamos, realiza las siguientes exploraciones:

### Parte 1 — Tu animal favorito
1. Elige tu animal favorito en inglés (e.g., `"dolphin"`, `"tiger"`, `"eagle"`)
2. Obtén su vector: `v_animal = model["nombre_del_animal"]`
3. Busca los 10 tokens más similares
4. ¿Los resultados tienen sentido semántico? ¿Encuentras tokens sorprendentes?

### Parte 2 — Operaciones vectoriales
Diseña operaciones análogas a `king - man + woman` con tu animal:
- `"dog" - "domestic" + "wild"` → ¿a qué llega?
- `"eagle" - "fly" + "swim"` → ¿qué predice el modelo?
- Inventa tu propia operación semántica y explica qué esperabas y qué obtuviste

### Parte 3 — Visualización
Toma tu animal, sus vecinos más cercanos y alguna operación vectorial, y grafícalos en 3D con PCA igual que hicimos arriba. ¿Forman clusters semánticos con sentido?

> **Tip:** Si una palabra no está en el vocabulario, el modelo lanzará un `KeyError`. Verifica con `"palabra" in model.key_to_index` antes de usarla.

### Resolución del reto: hipótesis previas

Elegimos **tiger** y esperamos asociaciones con felinos o animales. Un corpus de noticias también puede asociar nombres propios y otros sentidos. La lista de referencia previa es descriptiva, no una clasificación infalible.

Calculamos `tiger - wild + domestic` (expectativa: `cat`) y `puppy - dog + cat` (expectativa: `kitten`). El código informa candidatos, posición del esperado y coseno cuando existe. La PCA incluye operación y objetivo; los índices remiten a etiquetas laterales.

In [ ]:
REFERENCIA_ANIMAL = {
    "lion", "leopard", "jaguar", "cheetah", "panther", "cat",
    "tigers", "lions", "leopards", "felid", "feline",
}


def analizar_modelo(modelo, origen):
    ausentes_animal = faltantes(modelo, [ANIMAL])
    if ausentes_animal:
        raise KeyError(f"El animal {ANIMAL!r} no está en el vocabulario; elige otro.")
    vecinos = vecinos_coseno(
        modelo, modelo[ANIMAL], topn=TOP_N, excluir=[ANIMAL], bloque=BLOQUE_BUSQUEDA
    )
    nombres = [r["palabra"] for r in vecinos]
    coincidencias = [p for p in nombres if p in REFERENCIA_ANIMAL]
    observacion = (
        f"Se obtuvieron {len(vecinos)} vecinos de {ANIMAL}. "
        f"{len(coincidencias)} coinciden exactamente con la lista de referencia "
        f"de felinos fijada antes de ejecutar: {coincidencias}. "
        "Los restantes pueden ser asociaciones válidas, variantes, nombres propios "
        "o sesgos del corpus; su ausencia en esa lista no demuestra un error."
    )
    analogias = []
    for especificacion in ANALOGIAS:
        item = dict(especificacion)
        fuentes = [*item["positivos"], *item["negativos"]]
        item["faltantes"] = faltantes(modelo, fuentes)
        if item["faltantes"]:
            item.update({"estado": "omitida_vocabulario", "vecinos": []})
        else:
            consulta = vector_analogia(modelo, item["positivos"], item["negativos"])
            candidatos = vecinos_coseno(
                modelo, consulta, topn=TOP_N, excluir=fuentes, bloque=BLOQUE_BUSQUEDA
            )
            item.update({"estado": "calculada", "vecinos": candidatos})
            item["posicion_esperado_top10"] = next(
                (i for i, fila in enumerate(candidatos, start=1)
                 if fila["palabra"] == item["esperado"]), None
            )
            item["esperado_en_vocabulario"] = item["esperado"] in modelo.key_to_index
            item["coseno_esperado"] = (
                coseno(consulta, modelo[item["esperado"]])
                if item["esperado_en_vocabulario"] else None
            )
            if item["posicion_esperado_top10"] is not None:
                item["observacion"] = (
                    f"La palabra esperada {item['esperado']} apareció en la posición "
                    f"{item['posicion_esperado_top10']} de los candidatos filtrados."
                )
            else:
                item["observacion"] = (
                    f"La palabra esperada {item['esperado']} no apareció entre "
                    f"los {len(candidatos)} candidatos; esto no prueba que la relación "
                    "sea inexistente en todo el espacio."
                )
        analogias.append(item)
    return {
        "origen": origen, "animal": ANIMAL, "top_n_solicitado": TOP_N,
        "vecinos_animal": vecinos, "referencia_previa": sorted(REFERENCIA_ANIMAL),
        "observacion_animal": observacion, "analogias": analogias,
        "metodo": "Aritmética original; coseno normalizado; exclusión de palabras fuente.",
    }


In [ ]:
def preparar_pca(modelo, palabras, analogia=None):
    palabras = list(dict.fromkeys(palabras))
    palabras = [p for p in palabras if p in modelo.key_to_index]
    etiquetas = list(palabras)
    vectores = [np.asarray(modelo[p], dtype=np.float64) for p in palabras]
    tipos = ["palabra"] * len(palabras)
    if analogia is not None and analogia["estado"] == "calculada":
        etiquetas.append("RESULTADO: " + analogia["expresion"])
        vectores.append(vector_analogia(
            modelo, analogia["positivos"], analogia["negativos"]
        ))
        tipos.append("operacion")
        if analogia["esperado"] in palabras:
            tipos[palabras.index(analogia["esperado"])] = "esperado"
    if ANIMAL in palabras:
        tipos[palabras.index(ANIMAL)] = "animal"
    coordenadas, explicada = pca_tres_dimensiones(vectores)
    return {
        "etiquetas": etiquetas, "tipos": tipos, "coordenadas": coordenadas,
        "varianza_explicada": explicada, "dimensiones_originales": int(modelo.vector_size),
        "componentes_pca": 3,
    }


def crear_grafico_pca(datos, titulo):
    figura = plt.figure(figsize=(13, 7.5))
    rejilla = figura.add_gridspec(1, 2, width_ratios=[2.2, 1])
    eje = figura.add_subplot(rejilla[0, 0], projection="3d")
    lateral = figura.add_subplot(rejilla[0, 1])
    lateral.axis("off")
    estilos = {
        "palabra": ("#2166ac", "o", 45),
        "animal": ("#d97706", "o", 110),
        "esperado": ("#27814b", "D", 95),
        "operacion": ("#b52b54", "*", 220),
    }
    coordenadas = datos["coordenadas"]
    margen = max(float(np.ptp(coordenadas, axis=0).max()), 1e-3) * 0.012
    for indice, (etiqueta, tipo, punto) in enumerate(
        zip(datos["etiquetas"], datos["tipos"], coordenadas), start=1
    ):
        color, marcador, tamano = estilos[tipo]
        eje.scatter(*punto, color=color, marker=marcador, s=tamano, depthshade=False)
        desplazamiento = margen * (1 if indice % 2 else -1)
        eje.text(*(punto + desplazamiento), str(indice), fontsize=9)
    for indice in range(3):
        porcentaje = datos["varianza_explicada"][indice] * 100
        getattr(eje, f"set_{'xyz'[indice]}label")(f"PC{indice + 1} ({porcentaje:.1f}%)")
    eje.view_init(elev=20, azim=35)
    lateral.text(0, 1, "Índice de puntos", va="top", weight="bold", fontsize=12)
    for indice, (etiqueta, tipo) in enumerate(zip(datos["etiquetas"], datos["tipos"]), start=1):
        # El resultado se muestra en dos líneas para no salir del margen.
        texto = f"{indice}. {etiqueta}".replace("RESULTADO: ", "RESULTADO:\n    ")
        lateral.text(0, 0.95 - (indice - 1) * 0.046, texto, va="top",
                     fontsize=9, color=estilos[tipo][0])
    figura.suptitle(titulo, fontsize=15, weight="bold", y=0.98)
    figura.text(
        0.5, 0.025,
        f"Dimensión original: {datos['dimensiones_originales']} · PCA: 3 componentes · "
        f"Varianza conservada: {datos['varianza_explicada'].sum():.1%}\n"
        "Estrella: vector calculado; rombo: palabra esperada. Coseno medido en el espacio original.",
        ha="center", fontsize=9
    )
    figura.subplots_adjust(left=0.02, right=0.98, bottom=0.11, top=0.9, wspace=0.12)
    return figura


In [ ]:
def guardar_csv(ruta, filas, campos):
    with ruta.open("w", encoding="utf-8-sig", newline="") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(filas)


def exportar_resultados(resultado, proyecciones):
    identificador = re.sub(r"[^A-Za-z0-9_.-]", "_", resultado["origen"]["modelo"])
    ejecucion = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid4().hex[:8]
    carpeta = DIRECTORIO_SALIDA / identificador / ejecucion
    carpeta.mkdir(parents=True, exist_ok=False)
    guardar_csv(
        carpeta / "vecinos_animal.csv", resultado["vecinos_animal"], ["palabra", "coseno"]
    )
    filas_analogias = [
        {"analogia": item["expresion"], "posicion": posicion,
         "palabra": fila["palabra"], "coseno": fila["coseno"]}
        for item in resultado["analogias"]
        for posicion, fila in enumerate(item["vecinos"], start=1)
    ]
    guardar_csv(carpeta / "analogias.csv", filas_analogias,
                ["analogia", "posicion", "palabra", "coseno"])
    exportable = dict(resultado)
    exportable["ejecucion"] = ejecucion
    exportable["pca"] = {}
    for nombre, datos in proyecciones.items():
        filas = [
            {"etiqueta": etiqueta, "tipo": tipo,
             "pc1": float(punto[0]), "pc2": float(punto[1]), "pc3": float(punto[2])}
            for etiqueta, tipo, punto in zip(
                datos["etiquetas"], datos["tipos"], datos["coordenadas"]
            )
        ]
        guardar_csv(carpeta / f"pca_{nombre}.csv", filas,
                    ["etiqueta", "tipo", "pc1", "pc2", "pc3"])
        exportable["pca"][nombre] = {
            "dimensiones_originales": datos["dimensiones_originales"],
            "componentes": datos["componentes_pca"],
            "varianza_explicada": datos["varianza_explicada"].tolist(),
            "puntos": filas,
        }
        figura = crear_grafico_pca(datos, f"{resultado['origen']['modelo']} · {nombre}")
        try:
            figura.savefig(carpeta / f"pca_{nombre}.png", dpi=160, bbox_inches="tight")
        finally:
            plt.close(figura)
    (carpeta / "resultados.json").write_text(
        json.dumps(exportable, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8"
    )
    lineas = [
        "# Resultados ejecutados: embeddings",
        "",
        f"Modelo: **{resultado['origen']['modelo']}**.",
        f"Ejecución: {ejecucion}.",
        f"Procedencia: {resultado['origen']['procedencia']}.",
        f"Dimensiones: {resultado['origen']['dimensiones_embedding']}.",
        f"Vocabulario: {resultado['origen']['vocabulario']}.",
        "",
        "## Vecinos del animal",
        "",
        resultado["observacion_animal"],
        "",
        "| Posición | Token | Coseno |",
        "|---:|---|---:|",
    ]
    lineas += [
        f"| {i} | {fila['palabra']} | {fila['coseno']:.6f} |"
        for i, fila in enumerate(resultado["vecinos_animal"], start=1)
    ]
    for item in resultado["analogias"]:
        lineas += ["", f"## {item['expresion']}", "", item["hipotesis"]]
        if item["estado"] != "calculada":
            lineas += ["Omitida: fuera del vocabulario: " + ", ".join(item["faltantes"])]
            continue
        lineas += [
            "", item["observacion"],
            f"Coseno con {item['esperado']}: {item['coseno_esperado']}.",
            "", "| Posición | Token | Coseno |", "|---:|---|---:|",
        ]
        lineas += [
            f"| {i} | {fila['palabra']} | {fila['coseno']:.6f} |"
            for i, fila in enumerate(item["vecinos"], start=1)
        ]
    lineas += ["", "## PCA y límites", ""]
    for nombre, datos in proyecciones.items():
        lineas += [
            f"- {nombre}: 3 componentes; varianza conservada "
            f"{datos['varianza_explicada'].sum():.2%}; "
            f"![PCA {nombre}](pca_{nombre}.png)"
        ]
    lineas += [
        "",
        "Los vecinos reflejan asociaciones del corpus, no una clasificación semántica "
        "ni probabilidades. PCA puede distorsionar distancias. La coincidencia con una "
        "hipótesis no garantiza que la analogía generalice.",
        "",
        "Fuente del reto: https://github.com/EAFIT-IA/si3003-artificial-intelligence/"
        "blob/main/notebooks/Lecture9/tokenization_Introduction.ipynb",
    ]
    (carpeta / "reporte.md").write_text("\n".join(lineas), encoding="utf-8")
    return carpeta


In [ ]:
# === RETO COMPLETADO EN SU CELDA ORIGINAL ===
# Parte 1: animal en inglés, diez vecinos y comentario basado en resultados.
mi_animal = ANIMAL
resultado = analizar_modelo(model, origen) if model is not None else None
if resultado is None:
    print("Reto implementado; ejecución empírica pendiente de cargar el modelo.")
else:
    print(f"Animal: {mi_animal}; dimensión del embedding: {model.vector_size}")
    for i, fila in enumerate(resultado["vecinos_animal"], 1):
        print(f"{i:2d}. {fila['palabra']:25s} coseno={fila['coseno']:.4f}")
    print(resultado["observacion_animal"])

# Parte 2: operación del animal y analogía propia, sin presuponer sus respuestas.
if resultado is not None:
    for item in resultado["analogias"]:
        print("\nOperación:", item["expresion"])
        print("Expectativa previa:", item["hipotesis"])
        if item["estado"] != "calculada":
            print("Omitida por vocabulario:", item["faltantes"])
            continue
        for i, fila in enumerate(item["vecinos"], 1):
            print(f"{i:2d}. {fila['palabra']:25s} coseno={fila['coseno']:.4f}")
        print(item["observacion"])
        print("Coseno con el esperado:", item["coseno_esperado"])

# Parte 3: PCA por figura con el vector de la operación y el objetivo.
proyecciones = {}
if resultado is not None:
    analogia_animal = resultado["analogias"][0]
    palabras = [mi_animal, *[r["palabra"] for r in resultado["vecinos_animal"]],
                *analogia_animal["positivos"], *analogia_animal["negativos"],
                analogia_animal["esperado"]]
    proyecciones["animal"] = preparar_pca(model, palabras, analogia_animal)
    propia = resultado["analogias"][1]
    if propia["estado"] == "calculada":
        palabras = [*propia["positivos"], *propia["negativos"], propia["esperado"],
                    *[r["palabra"] for r in propia["vecinos"][:3]]]
        proyecciones["analogia_propia"] = preparar_pca(model, palabras, propia)
    for nombre, datos in proyecciones.items():
        figura = crear_grafico_pca(datos, f"{origen['modelo']} · {nombre}")
        plt.show()
        plt.close(figura)
    carpeta_generada = exportar_resultados(resultado, proyecciones)
    print("CSV, PNG, JSON e informe de esta ejecución:", carpeta_generada.resolve())
else:
    print("No se exportan vecinos ni métricas inventados.")

## Reflexión de la solución y estado real

- **Tokenización frente a embeddings:** dividir texto en unidades e identificar un vector para cada unidad son procesos distintos. Este ejercicio consulta entradas existentes de Word2Vec; no demuestra cómo un LLM convierte palabras nuevas en subpalabras.
- **Vector estático:** una entrada tiene el mismo vector aunque cambie la oración. Por ello puede mezclar sentidos y asociaciones; un embedding contextual puede representar usos distintos según el contexto.
- **Expectativa y evidencia:** las dos analogías son hipótesis lingüísticas previas. El informe registra resultados y posiciones efectivamente obtenidos. Que `cat` o `kitten` no aparezcan entre los diez primeros no autoriza a inventar una explicación causal; que aparezcan tampoco garantiza una regla universal.
- **Corpus y sesgos:** Google News representa lenguaje de noticias de un corpus particular. Frecuencia, época, nombres propios, estereotipos y cobertura cultural influyen en los vecinos. GloVe entrena con otro método y corpus: los resultados no son intercambiables.
- **Vecinos sorprendentes:** la lista de referencia solo ayuda a contrastar una expectativa explícita sobre felinos. Un término fuera de ella puede ser semánticamente pertinente. Para justificar una interpretación más fuerte habría que examinar usos reales del término y comparar más consultas; el coseno por sí solo no basta.
- **PCA:** tres coordenadas son una proyección del conjunto elegido, no las dimensiones originales. La varianza explicada cuantifica lo conservado para ese conjunto. Cambiar los puntos cambia los ejes; no conviene comparar coordenadas de PCA ajustados por separado como si fueran un mismo espacio.
- **Vocabulario:** las ausencias se reportan. Convertir arbitrariamente a minúsculas, fabricar ceros o sustituir palabras sin anotarlo cambiaría el experimento.
- **Alcance:** no se entrenó un nuevo Word2Vec ni se midió accuracy de un clasificador. Se exploraron vectores preentrenados mediante consultas reproducibles.

## Fuentes

- [Guía original EAFIT-IA y reto](https://github.com/EAFIT-IA/si3003-artificial-intelligence/blob/main/notebooks/Lecture9/tokenization_Introduction.ipynb).
- [Gensim KeyedVectors: formatos, acceso a vectores y similitud](https://radimrehurek.com/gensim/models/keyedvectors.html).
- [Gensim Downloader](https://radimrehurek.com/gensim/downloader.html).
- [Catálogo y metadatos de gensim-data](https://github.com/RaRe-Technologies/gensim-data).

**Estado de la entrega:** implementación completa y pruebas offline incluidas. La evidencia empírica de Google News se obtiene al ejecutar con el modelo oficial; no se ha sustituido por datos sintéticos ni por GloVe.

### Cambios sobre el original

Markdown intacto, celdas originales en el mismo orden y etiquetadas con `guia_indice_original`. Instalación/carga explícitas y guardas en ejemplos; reto completado en su posición. Funciones y pruebas añadidas. CSV, PNG, JSON e informe solo al ejecutar un modelo, en rutas únicas. Salidas y contadores limpiados. No se descargó Google News ni se afirman resultados nuevos. Validación offline: sintaxis, preservación de estructura, funciones y ejecución predeterminada sin red.